**AGENTIC AI - IMPROVING SQL GENERATION WITH REFLECTION**

In [10]:
import json

import pandas as pd
from dotenv import load_dotenv

import utils as utils

_ = load_dotenv()

aisuite ci da un modo unico per interagire con diversi LLM, cosi' possiamo evitare di preoccuparci del setup di ogni singolo modello

In [11]:
import aisuite

client = aisuite.Client()


Setup del Database (SQLite automatically filled)

**Schema overview**:

• id → unique event ID (autoincrement).
• product_id, product_name, brand, category, color → identify the product.
• action → type of event (insert, restock, sale, price_update).
• qty_delta → stock change (+ for insert/restock, – for sale, 0 for price updates).
• unit_price → price at that moment (NULL for restock).
• notes → optional description of the event.
• ts → timestamp when the event was logged.

In [12]:
utils.create_transactions_db("products.db")

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


In [13]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the users's question,
    write a SQL query for SQLite.
    
    Schema:
    {schema}
    
    User question:
    {question}
    
    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

# Schema as a string
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# Ask a question about the data in natural language
question = "Which color of product has the highest total sales?"
sql_v1 = generate_sql(question, schema, "openai:gemini-3.5-flash-lite")

utils.print_html(sql_v1)

In [14]:

# Execute the generated query sql (sql_v1) against the products.db database
# the results are returned as pandas DataFrame
df_sql_v1 = utils.execute_sql(sql_v1, db_path="products.db")

utils.print_html(df_sql_v1, title="Output of the SQL Query V1")


color
blue
